![DB Academy](https://files.training.databricks.com/binder/prod_main/ai-bi-for-data-analysts-en_us-2.0.2/images/20261005T110950Z/AI BI for Data Analysts/Includes/images/common/db-academy.png)

# Demo - REQUIRED - Data Setup and Exploration

## Overview

This is the **required setup notebook** for the AI/BI for Data Analysts course. It does two things.

1. **Sets up the Canada Sales data** in your personal catalog. Tables, volume, and source files. The setup is safe to re-run.

2. **Walks through the tables** with a *"just enough SQL"* refresher. The handful of SQL building blocks you'll use to define dashboard datasets throughout the rest of the course.

The **Canada Sales** dataset is a fictional Canadian B2B business with five tables covering products, customers, orders, sales opportunities, and yearly sales targets.

## Learning Objectives

By the end of this demo, you will be able to:

1. **Run the required setup script** to create the course tables in your catalog.
2. **Identify the five tables** and understand the role of each one.
3. **Use the core SQL building blocks** needed for dashboard datasets: aggregations (`SUM`, `COUNT`, `AVG`), `GROUP BY`, and `JOIN`.
4. **Write the main orders join** combining **ca_orders**, **ca_customers**, and **ca_products**. This query reappears as a dashboard dataset in the next demo.

## REQUIRED - SELECT A COMPUTE ENVIRONMENT

<div style="
  border-left: 4px solid #f44336;
  background: #ffebee;
  padding: 14px 18px;
  border-radius: 4px;
  margin: 16px 0;
">
  <strong style="display:block; color:#c62828; margin-bottom:6px; font-size: 1.1em;">Select Serverless Compute</strong>
  <div style="color:#333;">

Before starting this notebook, select the required compute environment listed below.

- **Serverless Compute, Version 5**

![Serverless Select](https://files.training.databricks.com/binder/prod_main/ai-bi-for-data-analysts-en_us-2.0.2/images/20261005T110950Z/AI BI for Data Analysts/Includes/images/common/select-serverless.png)
<br></br>
  - How to select an environment version:
[AWS](https://docs.databricks.com/aws/en/compute/serverless/dependencies#-select-an-environment-version) |
[Azure](https://learn.microsoft.com/en-us/azure/databricks/compute/serverless/dependencies#-select-an-environment-version) |
[GCP](https://docs.databricks.com/gcp/en/compute/serverless/dependencies#-select-an-environment-version)

**NOTE:**  This notebook requires **SQL and Python, and was tested using Serverless V5**. All purpose compute may work but is not guaranteed to behave the same or support all features demonstrated.
  </div>
</div>


## A. Classroom Setup

1. Run the cell below. It will:

    - Create (or use) your personal catalog named **labuser_UNIQUE_ID**.
    - Create the **aibi_dashboards** schema.
    - Create a Unity Catalog volume named **source_files**.
    - Copy the course CSVs into the volume and build the five course tables.

In [0]:
%run ./Includes/Classroom-Setup-Required-Data

✅ Vocareum workspace detected.
✅ Using existing Vocareum catalog: 'labuser17404915_1791275222'.



  STEP 1: Verifying catalog exists: labuser17404915_1791275222
  Catalog 'labuser17404915_1791275222' exists.

  STEP 2: Setting up 1 schema(s) in catalog: labuser17404915_1791275222
  [1/1] Checking: `labuser17404915_1791275222`.`aibi_dashboards`... CREATED

  COMPLETE: 1 schema(s) created, 0 already existed.


  STEP 1: Verifying catalog exists: labuser17404915_1791275222
  Catalog 'labuser17404915_1791275222' exists.

  STEP 2: Setting up 1 schema(s) in catalog: labuser17404915_1791275222
  [1/1] Checking: `labuser17404915_1791275222`.`aibi_dashboards_taxi`... CREATED

  COMPLETE: 1 schema(s) created, 0 already existed.



✅ Volume ready: labuser17404915_1791275222.aibi_dashboards.source_files
✅ Volume ready: labuser17404915_1791275222.aibi_dashboards.business_icon



  Searching for 'Includes/data/demos' folder...
  Current directory: /Workspace/Users/labuser17404915_1791275222@vocareum.com/ai-bi-for-data-analysts-en_us-2.0.2/AI BI for Data Analysts
  Checking: /Workspace/Users/labuser17404915_1791275222@vocareum.com/ai-bi-for-data-analysts-en_us-2.0.2/AI BI for Data Analysts/Includes/data/demos... FOUND


  STEP 1: Validating source workspace folder...
  Source folder found: /Workspace/Users/labuser17404915_1791275222@vocareum.com/ai-bi-for-data-analysts-en_us-2.0.2/AI BI for Data Analysts/Includes/data/demos

  STEP 2: Checking target volume path...
  Target volume path already exists: /Volumes/labuser17404915_1791275222/aibi_dashboards/source_files

  STEP 3: Reading source files...
  Found 4 file(s) in source folder.

  STEP 4: Copying 4 file(s) to target volume...
  Source:      /Workspace/Users/labuser17404915_1791275222@vocareum.com/ai-bi-for-data-analysts-en_us-2.0.2/AI BI for Data Analysts/Includes/data/demos
  Destination: /Volumes/labus

DataFrame[]

DataFrame[num_affected_rows: bigint, num_inserted_rows: bigint]

DataFrame[num_affected_rows: bigint, num_inserted_rows: bigint]

DataFrame[num_affected_rows: bigint, num_inserted_rows: bigint]

Information,Value
Your Catalog:,labuser17404915_1791275222
Your Demonstration Schema:,aibi_dashboards
Your Lab Schema:,aibi_dashboards_taxi



<div style="
  border-left: 4px solid #1976d2;
  background: #e3f2fd;
  padding: 14px 18px;
  border-radius: 4px;
  margin: 16px 0;
">
  <strong style="display:block; color:#0d47a1; margin-bottom:6px; font-size: 1.1em;">
    Where does this data come from?
  </strong>
  <div style="color:#333;">

For this course, we assume an upstream pipeline has already produced clean, analytics-ready tables in Unity Catalog.

You start where most analysts start, at the <strong>served data</strong>, and your job is to turn it into dashboards.

  </div>
</div>

## B. Preview the Tables (5)

Let's get familiar with what's in each table before writing any dashboard SQL.

| Table | What it contains | Rows |
|-------|------------------|-----:|
| **ca_orders** (fact table)| Transactional fact table (orders from 2021 to 2024) | 15,536 |
| **ca_products** | Product catalog (id, name, list price) | 9 |
| **ca_customers** | Canadian B2B customers (name, city, province) | 40 |
| **ca_sales_targets** | Sales targets for years 2021-2024| 4 |
| **ca_opportunities** | Sales pipeline (opportunities by phase) | 342 |

Run the cell below to confirm your default catalog and schema.
  - **Catalog** : **labuser_UNIQUE_ID**
  - **Schema** : **aibi_dashboards**

In [0]:
-- Confirm the session defaults set by the setup script
SELECT
  current_catalog() AS `Default Catalog`,
  current_schema() AS `Default Schema`;

Default Catalog,Default Schema
labuser17404915_1791275222,aibi_dashboards



<div style="
  border-left: 4px solid #1976d2;
  background: #e3f2fd;
  padding: 14px 18px;
  border-radius: 4px;
  margin: 16px 0;
">
  <strong style="display:block; color:#0d47a1; margin-bottom:6px; font-size: 1.1em;">
    Unity Catalog Namespace Review
  </strong>
  <div style="color:#333;">

Databricks uses a three-level namespace: `catalog.schema.table`.

Instead of writing the full name every time like `labuser_UNIQUE_ID.aibi_dashboards.ca_orders`, your environment is already configured with defaults:

- **Catalog**: `labuser_UNIQUE_ID`
- **Schema**: `aibi_dashboards`

That means you can simply use:

```sql
SELECT *
FROM ca_orders
```
  </div>
</div>

### B1. Show all Tables in your Schema

1. List the tables in the current schema with `SHOW TABLES`.

    You should see all five tables:
      - **ca_customers**
      - **ca_opportunities**
      - **ca_orders**
      - **ca_products**
      - **ca_sales_targets**

In [0]:
-- List all tables in the current (aibi_dashboards) schema
SHOW TABLES IN aibi_dashboards;

database,tableName,isTemporary
aibi_dashboards,ca_customers,false
aibi_dashboards,ca_opportunities,false
aibi_dashboards,ca_orders,false
aibi_dashboards,ca_products,false
aibi_dashboards,ca_sales_targets,false


### B2. `ca_orders` (the fact table)

This is the **fact table** of the dataset and the table you will join against the most.

1. View the **ca_orders** table.

    Notice it contains:
      - a unique **orderid** column
      - **customerid** and **productid** as foreign keys to **ca_customers** and **ca_products**
      - an **orderdate** `DATE` column covering **2021** - **2024**
      - transactional details: **quantity**, **orderamt**, and **salesrep**

In [0]:
-- Preview the ca_orders table
SELECT *
FROM ca_orders
ORDER BY orderdate
LIMIT 20;

orderid,customerid,productid,orderdate,quantity,orderamt,salesrep
16106,NR3611,BC111d,2021-11-16,361,22844.08,Hannah
16101,MM0344,A802a,2021-11-16,366,3722.22,Shalini
16104,SC7962,BC111d,2021-11-16,351,22211.28,Ethan
16105,SV2287,W8931,2021-11-16,324,14651.28,Hannah
16098,NM1588,A802a,2021-11-16,361,3671.37,Hannah
16100,SC7962,A802a,2021-11-16,345,3508.65,Hannah
16097,CS5977,W8931,2021-11-16,326,14741.72,Sam
16103,TC0514,W8931,2021-11-16,326,14741.72,Ingrid
16096,SC9674,A802a,2021-11-16,343,3488.31,Ethan
16095,QS1139,R9S10,2021-11-16,545,23887.35,Ingrid


### B3. Tour the other four tables

1. Run the cell below to execute a `SELECT *` against each table.

  - **ca_products** (9 rows): product catalog. 
      - **productid** is the join key to `ca_orders.productid`. Other columns: **productname**, **listprice**.

  - **ca_customers** (40 rows): Canadian B2B customers. 
      - **customerid** is the join key to `ca_orders.customerid`. 
      - Other columns: **customername**, **city**, **province** (two-letter codes like ON, QC, BC).

  - **ca_sales_targets** (4 rows): yearly sales targets. 
      - Key columns: **target_year**, **sales_target**.

  - **ca_opportunities** (342 rows): sales pipeline records (not closed orders).
      - Notable column: **phase** indicates funnel stage (`0-Prospecting`, `1-Active`, `2-Closed Won`, `3-Closed Lost`).

#### How the Tables Connect

`ca_orders` is the **fact table** at the center. Three dimension tables join into it on the keys shown below. `ca_opportunities` stands alone (no join used in this course).
<br></br>
<div style="width: 100%; max-width: 1100px; margin: 0 auto;">
<svg viewBox="0 0 1100 520" xmlns="http://www.w3.org/2000/svg" font-family="sans-serif" style="width: 100%; height: auto; display: block;">
<defs>
<marker id="rel-arr" viewBox="0 0 10 10" refX="9" refY="5" markerWidth="7" markerHeight="7" orient="auto-start-reverse">
<path d="M 0 0 L 10 5 L 0 10 z" fill="#618794"/>
</marker>
</defs>
<text x="550" y="32" text-anchor="middle" fill="#1B5162" font-size="15" font-weight="700" letter-spacing="2">CANADA SALES TABLE RELATIONSHIPS</text>
<rect x="60" y="80" width="240" height="100" rx="8" fill="#FFFFFF" stroke="#1B5162" stroke-width="1.5"/>
<text x="180" y="112" text-anchor="middle" fill="#0b2026" font-size="15" font-weight="700">ca_customers</text>
<text x="180" y="138" text-anchor="middle" fill="#5E7077" font-size="12">DIMENSION (40 rows)</text>
<text x="180" y="162" text-anchor="middle" fill="#0b2026" font-size="13" font-weight="600">customerid (PK)</text>
<rect x="800" y="80" width="240" height="100" rx="8" fill="#FFFFFF" stroke="#1B5162" stroke-width="1.5"/>
<text x="920" y="112" text-anchor="middle" fill="#0b2026" font-size="15" font-weight="700">ca_products</text>
<text x="920" y="138" text-anchor="middle" fill="#5E7077" font-size="12">DIMENSION (9 rows)</text>
<text x="920" y="162" text-anchor="middle" fill="#0b2026" font-size="13" font-weight="600">productid (PK)</text>
<rect x="370" y="210" width="360" height="120" rx="8" fill="#F9F7F4" stroke="#FE3722" stroke-width="3"/>
<text x="550" y="240" text-anchor="middle" fill="#0b2026" font-size="16" font-weight="700">ca_orders</text>
<text x="550" y="262" text-anchor="middle" fill="#FE3722" font-size="13" font-weight="700">FACT TABLE (15,536 rows)</text>
<text x="550" y="288" text-anchor="middle" fill="#0b2026" font-size="13">orderid (PK), customerid (FK), productid (FK),</text>
<text x="550" y="308" text-anchor="middle" fill="#0b2026" font-size="13">orderdate, quantity, orderamt, salesrep</text>
<rect x="370" y="390" width="360" height="100" rx="8" fill="#FFFFFF" stroke="#1B5162" stroke-width="1.5"/>
<text x="550" y="422" text-anchor="middle" fill="#0b2026" font-size="15" font-weight="700">ca_sales_targets</text>
<text x="550" y="448" text-anchor="middle" fill="#5E7077" font-size="12">DIMENSION (4 rows)</text>
<text x="550" y="472" text-anchor="middle" fill="#0b2026" font-size="13" font-weight="600">target_year (PK)</text>
<rect x="800" y="390" width="240" height="100" rx="8" fill="#FFFFFF" stroke="#9E9E9E" stroke-width="1.5" stroke-dasharray="4,3"/>
<text x="920" y="422" text-anchor="middle" fill="#0b2026" font-size="15" font-weight="700">ca_opportunities</text>
<text x="920" y="448" text-anchor="middle" fill="#5E7077" font-size="12">STANDALONE (342 rows)</text>
<text x="920" y="472" text-anchor="middle" fill="#5E7077" font-size="11" font-style="italic">no join in this course</text>
<line x1="300" y1="155" x2="430" y2="215" stroke="#618794" stroke-width="2" marker-end="url(#rel-arr)"/>
<text x="308" y="200" fill="#1B5162" font-size="12" font-weight="600">customerid</text>
<line x1="800" y1="155" x2="670" y2="215" stroke="#618794" stroke-width="2" marker-end="url(#rel-arr)"/>
<text x="720" y="200" fill="#1B5162" font-size="12" font-weight="600">productid</text>
<line x1="550" y1="390" x2="550" y2="335" stroke="#618794" stroke-width="2" marker-end="url(#rel-arr)"/>
<text x="560" y="365" fill="#1B5162" font-size="12" font-weight="600">YEAR(orderdate) = target_year</text>
</svg>
</div>

In [0]:
SELECT *
FROM ca_products;

In [0]:
SELECT * 
FROM ca_customers;

In [0]:
SELECT * 
FROM ca_sales_targets;

In [0]:
SELECT * 
FROM ca_opportunities 
ORDER BY date;

## C. AI/BI Dashboard Architecture

Our goal is simple. Take the Unity Catalog tables you just previewed and turn them into a **shareable AI/BI Dashboard** that business users can use to gain insights into the data.

<br></br>

<div style="width: 100%; max-width: 1200px; margin: 0 auto;">

<svg viewBox="0 0 1200 540" xmlns="http://www.w3.org/2000/svg" font-family="sans-serif" style="width: 100%; height: auto; display: block;">
  <defs>
    <marker id="arch-arr" viewBox="0 0 10 10" refX="9" refY="5" markerWidth="7" markerHeight="7" orient="auto-start-reverse">
      <path d="M 0 0 L 10 5 L 0 10 z" fill="#618794"/>
    </marker>
  </defs>

  <!-- Left subgraph: Unity Catalog Sources -->
  <rect x="20" y="30" width="370" height="490" rx="10" fill="#F9F7F4" stroke="#1B5162" stroke-width="2"/>
  <text x="205" y="58" text-anchor="middle" fill="#1B5162" font-size="14" font-weight="700" letter-spacing="2">UNITY CATALOG DASHBOARD SOURCES</text>

  <!-- Source 1: FullOrdersData -->
  <rect x="50" y="85" width="310" height="140" rx="6" fill="#FFFFFF" stroke="#1B5162" stroke-width="1.5"/>
  <text x="205" y="112" text-anchor="middle" fill="#0b2026" font-size="15" font-weight="700">FullOrdersData</text>
  <text x="205" y="137" text-anchor="middle" fill="#5E7077" font-size="14" letter-spacing="1.5">TABLES JOINED:</text>
  <text x="205" y="162" text-anchor="middle" fill="#0b2026" font-size="14">ca_orders</text>
  <text x="205" y="185" text-anchor="middle" fill="#0b2026" font-size="14">ca_customers</text>
  <text x="205" y="207" text-anchor="middle" fill="#0b2026" font-size="14">ca_products</text>

  <!-- Source 2: SalesTargets -->
  <rect x="50" y="255" width="310" height="120" rx="6" fill="#FFFFFF" stroke="#1B5162" stroke-width="1.5"/>
  <text x="205" y="282" text-anchor="middle" fill="#0b2026" font-size="15" font-weight="700">SalesTargets</text>
  <text x="205" y="307" text-anchor="middle" fill="#5E7077" font-size="14" letter-spacing="1.5">TABLES JOINED:</text>
  <text x="205" y="332" text-anchor="middle" fill="#0b2026" font-size="14">ca_sales_targets</text>
  <text x="205" y="355" text-anchor="middle" fill="#0b2026" font-size="14">ca_orders</text>

  <!-- Source 3: Opportunities -->
  <rect x="50" y="405" width="310" height="90" rx="6" fill="#FFFFFF" stroke="#1B5162" stroke-width="1.5"/>
  <text x="205" y="438" text-anchor="middle" fill="#0b2026" font-size="15" font-weight="700">Opportunities</text>
  <text x="205" y="470" text-anchor="middle" fill="#0b2026" font-size="14">ca_opportunities</text>

  <!-- Dashboard (center) -->
  <rect x="520" y="225" width="180" height="100" rx="8" fill="#F9F7F4" stroke="#FE3722" stroke-width="3"/>
  <text x="610" y="263" text-anchor="middle" fill="#0b2026" font-size="15" font-weight="700">AI/BI Dashboard</text>
  <text x="610" y="290" text-anchor="middle" fill="#5E7077" font-size="14" font-style="italic">Canada Sales</text>

  <!-- Right subgraph: Consumers -->
  <rect x="830" y="30" width="370" height="490" rx="10" fill="#EEEDE9" stroke="#618794" stroke-width="1.25"/>
  <text x="1015" y="58" text-anchor="middle" fill="#1B5162" font-size="14" font-weight="700" letter-spacing="2">CONSUMERS</text>

  <!-- Consumer 1: Analysts -->
  <rect x="860" y="85" width="310" height="110" rx="6" fill="#FFFFFF" stroke="#02A36F" stroke-width="1.5"/>
  <text x="1015" y="148" text-anchor="middle" fill="#0b2026" font-size="16" font-weight="700">Analysts</text>

  <!-- Consumer 2: Sales Leaders -->
  <rect x="860" y="215" width="310" height="110" rx="6" fill="#FFFFFF" stroke="#02A36F" stroke-width="1.5"/>
  <text x="1015" y="278" text-anchor="middle" fill="#0b2026" font-size="16" font-weight="700">Sales Leaders</text>

  <!-- Consumer 3: Executives -->
  <rect x="860" y="345" width="310" height="110" rx="6" fill="#FFFFFF" stroke="#02A36F" stroke-width="1.5"/>
  <text x="1015" y="408" text-anchor="middle" fill="#0b2026" font-size="16" font-weight="700">Executives</text>

  <!-- Arrows: sources to dashboard -->
  <line x1="360" y1="155" x2="520" y2="247" stroke="#618794" stroke-width="2" marker-end="url(#arch-arr)"/>
  <line x1="360" y1="315" x2="520" y2="275" stroke="#618794" stroke-width="2" marker-end="url(#arch-arr)"/>
  <line x1="360" y1="450" x2="520" y2="303" stroke="#618794" stroke-width="2" marker-end="url(#arch-arr)"/>

  <!-- Arrows: dashboard to consumers -->
  <line x1="700" y1="247" x2="860" y2="140" stroke="#618794" stroke-width="2" marker-end="url(#arch-arr)"/>
  <line x1="700" y1="275" x2="860" y2="270" stroke="#618794" stroke-width="2" marker-end="url(#arch-arr)"/>
  <line x1="700" y1="303" x2="860" y2="400" stroke="#618794" stroke-width="2" marker-end="url(#arch-arr)"/>
</svg>

</div>


## D. Just Enough SQL Review

The rest of the course is about building dashboards.

Every dashboard widget is backed by **one SQL query** (or a metric view: global or local).

Here's the small set of SQL building blocks you'll lean on throughout the course.

### D1. Aggregate everything with `SUM`, `COUNT`, `AVG`

> **Scenario:** Your sales leader asks, *"How big is our business overall, total sales, total orders, and average order value?"* A single aggregation query answers all of it.

1. Run the query below to compute full-table totals across all 15,536 orders.

    Notice:
      - The result is **one row** even though the underlying table has thousands of rows.
        - `COUNT(*)` counts rows
        - `SUM(...)` adds up a column
        - `AVG(...)` averages it
        - `ROUND(..., 2)` trims the average to two decimals

> An **aggregation** without a `GROUP BY` collapses the **entire table into a single row**. This is perfect for headline KPIs like *total sales* or *total units sold*.

In [0]:
-- Full-table aggregation: total sales and total quantity across all years
SELECT
  COUNT(*)                AS total_orders,
  SUM(orderamt)           AS total_sales,
  SUM(quantity)           AS total_quantity_sold,
  ROUND(AVG(orderamt), 2) AS avg_order_value
FROM ca_orders;

### D2. Aggregate by category with `GROUP BY`

> **Scenario:** Your sales leader asks, *"How are sales trending year over year?"*

1. Run the query below to get total sales and total quantity sold, one row per year.

    Notice:
      - You get **one row per year** instead of one row per order.
      - `YEAR(orderdate)` appears in both the `SELECT` list and the `GROUP BY` clause.
      - `ORDER BY order_year` puts the years in chronological order (2021, 2022, 2023, 2024).


> `GROUP BY` runs the aggregation **once per category**. Use the `YEAR()` function to pull the year out of a `DATE` column so you can bucket orders by year.

In [0]:
-- Total sales and total quantity sold, by year
SELECT
  YEAR(orderdate) AS order_year,
  COUNT(*)        AS total_orders,
  SUM(orderamt)   AS total_sales,
  SUM(quantity)   AS total_quantity_sold
FROM ca_orders
GROUP BY YEAR(orderdate)
ORDER BY order_year;

### D3. Combine tables with a `JOIN`

### We will use these queries below in the dashboard as our data sources

<div style="
  border-left: 4px solid #1976d2;
  background: #e3f2fd;
  padding: 14px 18px;
  border-radius: 4px;
  margin: 16px 0;
">
  <strong style="display:block; color:#0d47a1; margin-bottom:6px; font-size: 1.1em;">
    What is a JOIN?
  </strong>
  <div style="color:#333;">

A **join** combines rows from two or more tables using matching keys.

Think of it as lining up related data across tables so you can analyze it together.

<br></br>
<img src="https://files.training.databricks.com/binder/prod_main/ai-bi-for-data-analysts-en_us-2.0.2/images/20261005T110950Z/AI BI for Data Analysts/Includes/images/common/common-sql.png" width="100%">
  </div>
</div>



#### D3.1. Full Orders Query (3 table join: `ca_orders`, `ca_customers`, `ca_products`)

> **Scenario:** A stakeholder wants to see every order with the **customer name**, **province**, and **product name** instead of raw IDs. The detail lives in three different tables, so you need a `JOIN`.

We will use this query in the next demo as one of our dashboard datasets.

1. Run the query below to enrich each order with:
   - Customer name
   - Province
   - Product name (instead of raw IDs)

   Notice:
   - **Aliases** make the query easier to read:
     - `ca_orders` → `o`
     - `ca_customers` → `c`
     - `ca_products` → `p`

   - We join using **customerid** and **productid**, the foreign keys from earlier.

   - The result keeps the **same number of rows as `ca_orders`**, since every order has a matching customer and product.

In [0]:
-- Enrich every order with customer and product detail
SELECT
  c.customername,
  c.province,
  o.orderdate,
  YEAR(o.orderdate) AS year,
  p.productname,
  o.orderamt,
  o.quantity,
  o.salesrep
FROM ca_orders AS o
  INNER JOIN ca_customers AS c ON o.customerid = c.customerid
  INNER JOIN ca_products AS p ON o.productid = p.productid;

#### D3.2.  Yearly Sales Targets (2 table join: `ca_orders`, `ca_sales_targets`)

> **Scenario:** Leadership asks, *"Did we hit our yearly sales targets, and by how much?"* Compare actual sales (from `ca_orders`) against the goals stored in `ca_sales_targets`.

1. Run the query below to compare **actual sales vs targets by year**.

   This query calculates:
   - Total actual sales per year  
   - Difference from the target (variance)  
   - Percentage difference (variance %)  
   - Whether the target was **hit or missed**

   Notice:

   - We join `ca_orders` to `ca_sales_targets` using the **year** of the order date.

   - `SUM(o.orderamt)` aggregates total sales for each year.

   - `ROUND()` keeps the numbers clean and readable.

   - The **variance** shows how far off we are from the target.

   - The **variance %** puts that difference into context.

   - A `CASE` statement labels each year as:
     - `HIT` when actual sales meet or exceed the target  
     - `MISS` when they fall short

   - `GROUP BY` is required because we are aggregating sales by year.



In [0]:
SELECT
  t.target_set_date,
  t.target_year,
  t.sales_target,
  ROUND(SUM(o.orderamt), 2)                       AS actual_sales,
  ROUND(SUM(o.orderamt) - t.sales_target, 2)      AS variance,
  ROUND((SUM(o.orderamt) - t.sales_target) / t.sales_target * 100, 2) AS variance_pct,
  CASE
    WHEN SUM(o.orderamt) >= t.sales_target THEN 'HIT'
    ELSE 'MISS'
  END  AS status,
  t.notes
FROM ca_sales_targets AS t
  INNER JOIN ca_orders AS o ON YEAR(o.orderdate) = t.target_year
GROUP BY t.target_set_date, t.target_year, t.sales_target, t.notes
ORDER BY t.target_year DESC;


<div style="
  border-left: 4px solid #1976d2;
  background: #e3f2fd;
  padding: 14px 18px;
  border-radius: 4px;
  margin: 16px 0;
">
  <strong style="display:block; color:#0d47a1; margin-bottom:6px; font-size: 1.1em;">
    Just Enough SQL
  </strong>
  <div style="color:#333;">

This is a <strong>focused refresher</strong>, not a full SQL course.

Working knowledge of SQL is a prerequisite for the course.

For a full list of SQL functions view the **Alphabetical list of built-in functions** [AWS](https://docs.databricks.com/aws/en/sql/language-manual/sql-ref-functions-builtin-alpha) | [Azure](https://learn.microsoft.com/en-us/azure/databricks/sql/language-manual/sql-ref-functions-builtin-alpha) | [GCP](https://docs.databricks.com/gcp/en/sql/language-manual/sql-ref-functions-builtin-alpha)

  </div>
</div>

## E. Use Genie Code to Write SQL!
<br></br>
<div style="width: 100%; font-family: sans-serif;">

<div style="background: #F9F7F4; border-radius: 10px; padding: 24px 28px; box-shadow: 0 2px 8px rgba(27,49,57,0.06); border-top: 6px solid #FF5F46;">
  <img src="https://files.training.databricks.com/binder/prod_main/ai-bi-for-data-analysts-en_us-2.0.2/images/20261005T110950Z/AI BI for Data Analysts/Includes/images/common/genie-code.png" style="height: 64px; margin-bottom: 10px;">
  <div style="font-size: 15pt; color: #0B2026; line-height: 1.7; margin-bottom: 16px;">
  Want help writing SQL? Ask Genie Code! Click on the genie icon <img src="https://files.training.databricks.com/binder/prod_main/ai-bi-for-data-analysts-en_us-2.0.2/images/20261005T110950Z/AI BI for Data Analysts/Includes/images/common/genie-icon.png" style="height: 32px; vertical-align: middle;"> and begin querying. For example, click the <strong>Copy</strong> button below and paste into <strong>Genie Code</strong>.
  </div>
  <div style="display: flex; align-items: center; gap: 10px; background: #fff; border: 1px solid #ddd; border-radius: 6px; padding: 10px 14px; font-size: 14pt; font-family: monospace; color: #0B2026;">
    <span id="genie-query" style="flex: 1;">Write a query that joins the ca_orders and ca_products tables from my aibi_dashboards schema.</span>
    <button onclick="
    var text = document.getElementById('genie-query').innerText;
    var ta = document.createElement('textarea');
    ta.value = text;
    ta.style.position = 'fixed';
    ta.style.opacity = '0';
    document.body.appendChild(ta);
    ta.select();
    document.execCommand('copy');
    document.body.removeChild(ta);
    this.innerText = 'Copied!';
    var btn = this;
    setTimeout(function(){ btn.innerText = 'Copy'; }, 2000);
    " style="background: #FF5F46; color: white; border: none; border-radius: 4px; padding: 4px 12px; font-size: 13pt; cursor: pointer; white-space: nowrap;">Copy</button>
  </div>

</div>

</div>



## Conclusion

In this demo, you:

1. Ran the required setup script to create your **labuser_UNIQUE_ID.aibi_dashboards** schema and all five course tables.
2. Previewed **ca_products**, **ca_customers**, **ca_orders**, **ca_opportunities**, and **ca_sales_targets** and identified each table's key columns.
3. Saw how the five tables flow into an **AI/BI Dashboard** and out to business consumers.
4. Worked through the "just enough SQL" building blocks: full-table aggregation, `GROUP BY` by year, and `JOIN`.
5. Wrote the **main orders + customers + products join**. This query reappears as a dashboard dataset in the next demo.

&copy; <span id="dbx-year"></span> Databricks, Inc. All rights reserved.
Apache, Apache Spark, Spark, the Spark Logo, Apache Iceberg, Iceberg, and the Apache Iceberg logo are trademarks of the <a href="https://www.apache.org/" target="_blank">Apache Software Foundation</a>.<br/><br/><a href="https://databricks.com/privacy-policy" target="_blank">Privacy Policy</a> | <a href="https://databricks.com/terms-of-use" target="_blank">Terms of Use</a> | <a href="https://help.databricks.com/" target="_blank">Support</a>
<script>
  document.getElementById("dbx-year").textContent = new Date().getFullYear();
</script>